# Train small-lm on a free Colab GPU

Runs `data.py` / `train.py` / `plot_loss.py` from the repo unmodified, on a free T4.

**Before you start:** Runtime menu -> Change runtime type -> **T4 GPU**.

**Free-tier realities this notebook works around:**
- Colab's local disk is wiped every time the session disconnects/recycles (idle timeout, or the ~12h hard cap). So data and checkpoints live on **Google Drive** instead, and training runs with `--resume` so a disconnect costs you nothing but time.
- Free GPUs get pre-empted under load. Just re-run the last two cells when that happens -- `--resume` picks up from `out/ckpt.pt`.
- Keep the browser tab open/active; idle detection disconnects background tabs.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything persistent (data + checkpoints) lives under this Drive folder
# so it survives a disconnect. Only the repo code is re-cloned each session.
PROJECT_DIR = '/content/drive/MyDrive/small-lm'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print('persistent project dir:', PROJECT_DIR)

## Get the code

Replace the URL with your GitHub repo once you've pushed it. If you haven't pushed yet,
use the fallback cell below to upload a zip instead.

In [ ]:
REPO_URL = 'https://github.com/eric-solak/musicGPT.git'

%cd /content
!git clone $REPO_URL musicGPT
%cd musicGPT

In [ ]:
# Build the dataset directly onto Drive, so it only needs downloading once ever.
DATA_DIR = f'{PROJECT_DIR}/data'

import os
if os.path.exists(f'{DATA_DIR}/train.bin'):
    print('data already built, skipping')
else:
    !python data.py --source enwiki --parts 4 --data-dir "$DATA_DIR"

## Train

Run this cell. If the session disconnects, just re-run it -- `--resume` continues from
the last checkpoint on Drive instead of starting over.

In [ ]:
OUT_DIR = f'{PROJECT_DIR}/out'

resume_flag = '--resume' if os.path.exists(f'{OUT_DIR}/ckpt.pt') else ''
!python train.py --model base --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" \
    --batch-size 16 --grad-accum-steps 8 --eval-interval 50 {resume_flag}

## Check progress

Safe to run any time, including from a second Colab tab while training runs in the first.

In [ ]:
!python plot_loss.py --out-dir "$OUT_DIR"
from IPython.display import Image, display
display(Image(f'{OUT_DIR}/loss_curve.png'))

In [ ]:
!python sample.py --ckpt "$OUT_DIR/ckpt_best.pt" --prompt "The electric guitar" --num-samples 2

## When training is done

Everything you need is on Drive at `PROJECT_DIR`:
- `out/ckpt_best.pt` -- the checkpoint to deploy (see `space/` in the repo for the demo app)
- `out/metrics.csv`, `out/loss_curve.png` -- for the README's Results section
- `out/samples/step_*.txt` -- generations over time, for the before/after in the README

Download `ckpt_best.pt`, `metrics.csv`, `loss_curve.png`, and a couple of `samples/*.txt`
from Drive to your local repo, commit them, and follow `space/README.md` to deploy the demo.